In [1]:
import pandas as pd
import numpy as np
import altair as alt

In [14]:
from pathlib import Path
import re

# Normalise city names across datasets
city_fixes = {
    "Barletta-Andria-Trani": "Barletta",
    "Bolzano / Bozen": "Bolzano/Bozen",
    "Carbonia-Iglesias": "Carbonia",
    "Forlì-Cesena": "Forlì",
    "Forl�-Cesena": "Forlì",
    "Massa-Carrara": "Massa",
    "Medio Campidano": "Sanluri",
    "Monza e della Brianza": "Monza",
    "Ogliastra": "Tortolì",
    "Olbia-Tempio": "Olbia",
    "Pesaro e Urbino": "Pesaro",
    "Sud Sardegna": "Carbonia",
    "Verbano-Cusio-Ossola": "Verbania",
    "Valle d'Aosta / Vallée d'Aoste": "Aosta",
    "Valle d\"Aosta / Vallée d\"Aoste": "Aosta",
    "Valle d’Aosta / Vallée d’Aoste": "Aosta",
    "�Valle d\"�Aosta / Vall�e d\"�Aoste�": "Aosta",
    "Valle d\"Aosta/Vallée d\"Aoste": "Aosta",
    "Valle d\"Aosta/Vall�e d\"Aoste": "Aosta",
    "Valle d\"'Aosta/Vall�e d\"'Aoste": "Aosta",
    "Tortol�": "Tortolì",
}

def normalize_city(name: str) -> str:
    if pd.isna(name):
        return None
    s = str(name)
    s = s.replace("\ufeff", "").strip()
    s = s.strip("'\"")
    s = s.replace("’", "'")
    s = re.sub(r"\s*/\s*", "/", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s)
    s = city_fixes.get(s, s)
    if "valle" in s.lower() and "aosta" in s.lower():
        s = "Aosta"
    return s

# Aggregate municipal accident data across 15 yearly city files
# Assumes files named "Road accidents with injuries ... (cities) (x).csv" with x=1..15 in data/.
data_dir = Path("c:/Users/juanx/Documents/GitHub/juanxgi83.github.io/data")
city_files = sorted(data_dir.glob("Road accidents with injuries* (cities) (*.csv"))
if len(city_files) != 15:
    raise FileNotFoundError(f"Expected 15 city files, found {len(city_files)}")

raw_parts = []
for fp in city_files:
    part = pd.read_csv(fp)
    raw_parts.append(part)
raw = pd.concat(raw_parts, ignore_index=True)

# Keep descriptive columns and the observation
col_map = {
    "Territory": "city",
    "Localization of the accident": "localization",
    "Intersection (DESC)": "intersection",
    "Road accident type": "accident_type",
    "Deadly accident": "deadly",  # categorical flag
    "Road accident hour": "hour_band",
    "Week day": "weekday",
    "Month (DESC)": "month",
    "Observation": "value",
}
needed = list(col_map.keys())
missing = [c for c in needed if c not in raw.columns]
if missing:
    raise KeyError(f"Missing expected columns: {missing}")

acc = raw[needed].rename(columns=col_map)
acc["value"] = pd.to_numeric(acc["value"], errors="coerce").fillna(0)
acc["city"] = acc["city"].apply(normalize_city)
acc = acc.dropna(subset=["city"])

# Build compact, descriptive variable labels (skip dims that are "Total")
label_parts = [
    ("localization", "loc"),
    ("intersection", "intersection"),
    ("accident_type", "type"),
    ("deadly", "fatality"),
    ("hour_band", "hour"),
    ("weekday", "weekday"),
    ("month", "month"),
]

def make_label(row):
    parts = []
    for col, short in label_parts:
        val = str(row[col]).strip()
        if val.lower() == "total":
            continue
        parts.append(f"{short}={val}")
    return " | ".join(parts) if parts else None

acc["variable"] = acc.apply(make_label, axis=1)
acc = acc.dropna(subset=["variable"])  # drop all_categories rows

# Sum across the 15 years by city and variable, then pivot wide
agg = acc.groupby(["city", "variable"], as_index=False)["value"].sum()
wide = (
    agg
    .pivot(index="city", columns="variable", values="value")
    .fillna(0)
)

# Keep only three-dimension combos and the city total
cols = list(wide.columns)
three_dim = [c for c in cols if isinstance(c, str) and c.count(" | ") == 2]
wide = wide[sorted(three_dim)]
wide["city_total_15y"] = wide.sum(axis=1)

accidents_city_df = wide.reset_index()

# ---------------- Vehicle register (14 years) -----------------
veh_files = sorted(data_dir.glob("Vehicles - Public Register of Motor-vehicles - municipalities*(*.csv"))
veh_files = [fp for fp in veh_files if not str(fp).endswith("(15).csv")]  # keep 1..14
if len(veh_files) != 14:
    raise FileNotFoundError(f"Expected 14 vehicle files (excluding year 15), found {len(veh_files)}")

veh_parts = []
for fp in veh_files:
    part = pd.read_csv(fp)
    veh_parts.append(part)
veh_raw = pd.concat(veh_parts, ignore_index=True)

veh_map = {
    "Territory": "city",
    "Vehicle type": "vehicle_type",
    "TIME_PERIOD": "year",
    "Observation": "value",
}
veh_needed = list(veh_map.keys())
veh_missing = [c for c in veh_needed if c not in veh_raw.columns]
if veh_missing:
    raise KeyError(f"Missing expected vehicle columns: {veh_missing}")

veh = veh_raw[veh_needed].rename(columns=veh_map)
veh["value"] = pd.to_numeric(veh["value"], errors="coerce").fillna(0)
veh["city"] = veh["city"].apply(normalize_city)
veh = veh.dropna(subset=["city"])

# Aggregate per city-year-vehicle
veh_agg = veh.groupby(["city", "year", "vehicle_type"], as_index=False)["value"].sum()

# Compute Other Motor Cars at city-year level
motor_cars_label = "Motor cars"
euro_labels = [
    "Motor cars Euro 0",
    "Motor cars Euro 1",
    "Motor cars Euro 2",
    "Motor cars Euro 3",
    "Motor cars Euro 4",
    "Motor cars Euro 5",
    "Motor cars Euro 6",
]

veh_pivot = veh_agg.pivot_table(index=["city", "year"], columns="vehicle_type", values="value", aggfunc="sum")
veh_pivot = veh_pivot.fillna(0)

if motor_cars_label in veh_pivot.columns:
    euro_sum = veh_pivot.reindex(columns=euro_labels).sum(axis=1, min_count=1).fillna(0)
    veh_pivot["Other Motor Cars"] = (veh_pivot[motor_cars_label] - euro_sum).clip(lower=0)

veh_long = veh_pivot.reset_index().melt(id_vars=["city", "year"], var_name="vehicle_type", value_name="value")
veh_long = veh_long.dropna(subset=["value"])

veh_avg = veh_long.groupby(["city", "vehicle_type"], as_index=False)["value"].mean()

# Clean column names for wide pivot

def clean_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name.lower()

veh_avg["vehicle_col"] = veh_avg["vehicle_type"].apply(clean_name)
veh_wide = veh_avg.pivot(index="city", columns="vehicle_col", values="value").fillna(0)

# Add prefixed descriptive names
veh_wide = veh_wide.rename(columns={c: ("avg_" + c + "_14y") for c in veh_wide.columns})
if "avg_totale_14y" in veh_wide.columns:
    veh_wide = veh_wide.rename(columns={"avg_totale_14y": "avg_total_vehicles_14y"})

# Order vehicle columns: euro0-6, other motor cars, rest, total
veh_cols = list(veh_wide.columns)
order_keys = [
    "avg_motor_cars_euro_0_14y",
    "avg_motor_cars_euro_1_14y",
    "avg_motor_cars_euro_2_14y",
    "avg_motor_cars_euro_3_14y",
    "avg_motor_cars_euro_4_14y",
    "avg_motor_cars_euro_5_14y",
    "avg_motor_cars_euro_6_14y",
    "avg_other_motor_cars_14y",
]
ordered = [c for c in order_keys if c in veh_cols]
remaining = [c for c in veh_cols if c not in ordered and c != "avg_total_vehicles_14y"]
if "avg_total_vehicles_14y" in veh_cols:
    ordered += sorted(remaining) + ["avg_total_vehicles_14y"]
else:
    ordered += sorted(remaining)
veh_wide = veh_wide[ordered]

# Merge accidents + vehicles
combined_df = accidents_city_df.merge(veh_wide, on="city", how="left")

# Export to Excel
excel_out = data_dir / "municipal_accident_totals_15y.xlsx"
try:
    combined_df.to_excel(excel_out, index=False)
except PermissionError:
    print("Warning: could not write Excel (file locked)")

# Grand total across cities for accidents (city_total_15y)
city_total_grand = combined_df["city_total_15y"].sum()
print(city_total_grand)

# Sum of avg total vehicles across cities
if "avg_total_vehicles_14y" in combined_df.columns:
    total_avg_vehicles = combined_df["avg_total_vehicles_14y"].sum()
    print(total_avg_vehicles)

excel_out

combined_df.head()

2611504.0
15304792.42857143


,city,loc=Motorway | intersection=Bend | type=Accidents between vehicles,loc=Motorway | intersection=Bend | type=Accidents involving a single vehicle,loc=Motorway | intersection=Bend | type=Vehicle-pedestrian accident,loc=Motorway | intersection=Bump - slope - bottleneck | type=Accidents between vehicles,loc=Motorway | intersection=Bump - slope - bottleneck | type=Accidents involving a single vehicle,loc=Motorway | intersection=Bump - slope - bottleneck | type=Vehicle-pedestrian accident,loc=Motorway | intersection=Crossoroad | type=Accidents between vehicles,loc=Motorway | intersection=Straight stretch | type=Accidents between vehicles,loc=Motorway | intersection=Straight stretch | type=Accidents involving a single vehicle,...,avg_other_motor_cars_14y,avg_bus_or_trolley_bus_14y,avg_motor_cars_14y,avg_motor_units_14y,avg_motorcycles_14y,avg_other_vehicles_14y,avg_three_wheeler_or_motor_van_14y,avg_trailer_14y,avg_trucks_14y,avg_total_vehicles_14y
0,Agrigento,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,47.000000,162.857143,41673.357143,135.857143,11103.500000,0.0,333.571429,321.285714,4883.571429,58614.000000
1,Alessandria,146.0,262.0,1.0,0.0,0.0,0.0,0.0,898.0,463.0,...,99.500000,240.928571,58604.571429,144.285714,9359.857143,0.0,261.928571,269.500000,6810.428571,75691.500000
2,Ancona,87.0,88.0,3.0,2.0,1.0,1.0,0.0,331.0,161.0,...,123.857143,322.714286,61767.714286,98.214286,16111.214286,0.0,217.857143,224.714286,6604.500000,85346.928571
3,Aosta,35.0,74.0,1.0,0.0,1.0,0.0,0.0,94.0,81.0,...,816.357143,77.214286,125908.857143,48.500000,4126.142857,0.0,418.500000,102.500000,34840.785714,165522.500000
4,Arezzo,93.0,96.0,1.0,1.0,2.0,0.0,0.0,462.0,175.0,...,106.714286,632.857143,66873.357143,201.857143,14854.714286,0.0,556.571429,462.714286,8308.642857,91890.714286
